In [1]:
from ds_practice.posthog import run_hogql

In [ ]:
r = run_hogql(
    """
SELECT event, properties.$lib AS lib, count() AS n, uniq(person_id) AS users
FROM events
WHERE timestamp >= now() - INTERVAL 28 DAY
  AND (event = '$pageview' OR event LIKE 'billing.%')
GROUP BY event, lib
ORDER BY n DESC
LIMIT 15
""",
    name="wk04 smoke",
)
print(list(r))
print(r["columns"])
r["results"][:6]

['cache_key', 'cache_target_age', 'calculation_trigger', 'clickhouse', 'column_formats', 'columns', 'error', 'explain', 'hasMore', 'hogql', 'is_cached', 'last_refresh', 'limit', 'metadata', 'modifiers', 'next_allowed_client_refresh', 'offset', 'query', 'query_metadata', 'query_scan', 'query_status', 'resolved_compare_date_range', 'resolved_date_range', 'results', 'timezone', 'timings', 'types', 'used_data_warehouse_sources', 'warnings']
['event', 'lib', 'n', 'users']


[['$pageview', 'web', 5310, 2132],
 ['billing.xero_paid_sync_run', None, 3723, 1],
 ['billing.plan_expiry_left', None, 182, 7],
 ['billing.plan_expiry_review', None, 130, 5],
 ['billing.plans_viewed', 'web', 125, 54],
 ['billing.plan_expiry_deactivate', None, 84, 4]]

In [3]:
from ds_practice.posthog import read_sql

r = run_hogql(read_sql("../sql/events_28d.sql"), name="wk04 extract")
len(r["results"]), r["columns"]

(4961,
 ['event_id',
  'ts',
  'person_id',
  'event',
  'org_id',
  'plan',
  'host',
  'pathname'])

In [5]:
from ds_practice.events import extract_events

df = extract_events()
df.shape, df.schema

((4993, 8),
 Schema([('event_id', String),
         ('ts', Datetime(time_unit='us', time_zone='UTC')),
         ('person_id', String),
         ('event', String),
         ('org_id', Int64),
         ('plan', String),
         ('host', String),
         ('pathname', String)]))